# 📝 통계 기초 과제 LV2 정답 — 기술통계·상관·추론 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day08_기술통계_추론통계/data/` 입니다.
- 그래프 문제(4·7·10)는 자가채점이 없습니다 — 완성 그래프 이미지와 같은 모양이면 정답입니다.
- 서술 문제(1·4·5·6·7·9·10·11)에는 모범 서술을 함께 실었습니다(정답은 여럿).

아래 셀을 먼저 실행해 분석 라이브러리와 한글 폰트를 준비하세요.

In [ ]:
# [제공 코드] 통계 분석에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
print('행·열 크기:', df.shape)
print('\n[앞 5행] head()'); display(df.head())
print('\n[열·자료형·결측] info()'); df.info()
print('\n[수치 요약] describe()'); display(df.describe())
print('\n[범주형 요약] describe(exclude="number")'); display(df.describe(exclude='number'))

## 1. 데이터를 살펴보고 관찰 적기 (서술)
**배경**: 분석을 시작하기 전, 위 `데이터 살펴보기` 결과를 바탕으로 이 데이터가 어떤 데이터인지 스스로 정리해 봅니다.

**요구사항**:
- 위 `# [제공 코드]`(head·info·describe) 실행 결과를 보고, 아래 **서술 셀**에 관찰을 **2~3문장**으로 적으세요.
- 다음을 담으면 좋아요: 몇 개 나라의 몇 개 행인지, 어떤 열이 있는지, 결측치가 있는지, 지출과 기대수명이 대략 어느 방향으로 함께 움직이는지.

**관찰 (모범 서술 — 예시)**

이 데이터는 여섯 나라(캐나다·프랑스·독일·영국·일본·미국)의 1970~2020년 의료비 지출과 기대수명을 담은 274행 4열 표다. 열은 연도(`Year`)·국가(`Country`)·1인당 의료비 지출(`Spending_USD`)·기대수명(`Life_Expectancy`)이며 결측치는 없다. `describe` 를 보면 지출이 큰 나라·연도일수록 기대수명도 대체로 높아, 두 값이 같은 방향으로 움직이는 양의 관계가 있어 보인다.

## 2. 미국의 지출 요약 통계 (필터 + 기술통계)
**배경**: 나라마다 의료비 지출 수준과 그 변동 폭이 다릅니다. 먼저 **미국(USA)** 한 나라만 뽑아 지출(`Spending_USD`)의 대표값과 산포를 정리합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Country` 가 `'USA'` 인 행만 골라 그 `Spending_USD` 의 **평균**을 `usa_mean`, **표준편차**를 `usa_std` 에 담으세요(표준편차는 `.std()` 기본값 = 표본표준편차 ddof=1).
- **변동계수**를 `usa_cv` 에 담으세요. 변동계수 = 표준편차 ÷ 평균 × 100 (백분율, %).
- 세 값 모두 소수 **둘째 자리**까지 비교합니다.

**예시**
```
round(usa_mean, 2) → 4388.57
round(usa_std, 2)  → 3386.31
round(usa_cv, 2)   → 77.16
```
<details><summary>힌트</summary>

```text
접근방법:
- 한 나라만 보려면 불리언 조건으로 그 나라 행만 거른다. 그 뒤 평균·표준편차를 구하고, 변동계수는 둘로 계산한다.

세부구현:
1. Country 가 미국인 행만 불리언 인덱싱으로 고른다
2. 그 Spending_USD 의 평균과 표준편차를 각각 구한다
3. 표준편차를 평균으로 나눈 뒤 100을 곱해 변동계수(백분율)를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
usa = df[df['Country'] == 'USA']
usa_mean = usa['Spending_USD'].mean()
usa_std = usa['Spending_USD'].std()
# 변동계수(%) — 평균 대비 얼마나 흩어졌는지. 나라마다 지출 규모가 달라 표준편차만으로는 비교가 안 된다.
usa_cv = usa_std / usa_mean * 100
print(round(usa_mean, 2), round(usa_std, 2), round(usa_cv, 2))

In [ ]:
# [자가채점]  통계값은 정확일치 대신 허용오차(abs<tol)로 비교합니다.
assert abs(usa_mean - 4388.57) < 0.01
assert abs(usa_std - 3386.31) < 0.01
assert abs(usa_cv - 77.16) < 0.01, '변동계수 = 표준편차 ÷ 평균 × 100 이에요'
print("✅ 문제2 통과!")

### 해설 — 문제 2
- **접근법**: `df[df['Country'] == 'USA']` 로 미국 행만 거른 뒤 `.mean()`·`.std()` 로 대표값과 산포를 구합니다. 변동계수(CV)는 표준편차를 평균으로 나눠 **단위와 무관하게** 상대적 변동을 재는 값이라, 나라끼리 산포를 비교할 때 유용해요.
- **흔한 실수**: `.std()` 는 기본이 표본표준편차(ddof=1)입니다. `ddof=0`(모표준편차)으로 바꾸면 값이 조금 달라져 자가채점을 통과하지 못해요.
- **대안**: `usa['Spending_USD'].describe()` 로 평균·표준편차를 한 번에 확인한 뒤 변동계수만 따로 계산해도 됩니다.

## 3. 가장 최근 연도의 지출 1위·최하위 나라와 격차 (연도 필터 + 최댓값·최솟값·격차)
**배경**: 가장 최근 연도에 어느 나라가 1인당 의료비를 가장 많이·가장 적게 썼는지 찾고, **1위와 최하위의 격차**가 얼마나 벌어져 있는지까지 계산해 나라 간 지출 불균형을 확인합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- 데이터에서 **가장 최근 연도**를 `latest_year` 에 담으세요(`Year` 의 최댓값).
- 그 연도의 행만 골라, `Spending_USD` 가 **가장 큰** 나라 이름을 `top_country`(문자열)·그 지출 값을 `top_spending`, **가장 작은** 나라 이름을 `bottom_country`(문자열)·그 지출 값을 `bottom_spending` 에 담으세요.
- 1위와 최하위의 **지출 격차**(`top_spending - bottom_spending`)를 `spending_gap` 에 담으세요.
- `top_spending`·`bottom_spending`·`spending_gap` 은 소수 **둘째 자리**까지 비교합니다.

**예시**
```
latest_year             → 2020
top_country             → 'USA'
round(top_spending, 2)  → 11859.18
bottom_country          → 'Japan'
round(bottom_spending, 2) → 4665.64
round(spending_gap, 2)  → 7193.54
```
<details><summary>힌트</summary>

```text
접근방법:
- 최근 연도는 Year 열의 최댓값이다. 그 연도만 거른 뒤, 지출이 가장 큰 행과 가장 작은 행의 나라·값을 각각 뽑는다.
- 격차는 1위 지출에서 최하위 지출을 빼면 된다.

세부구현:
1. Year 의 최댓값을 구해 latest_year 에 담는다
2. Year 가 latest_year 인 행만 불리언 인덱싱으로 고른다
3. 그 안에서 Spending_USD 가 최대인 행(idxmax)과 최소인 행(idxmin)을 찾아 나라·지출 값을 꺼낸다
4. 1위 지출에서 최하위 지출을 빼 격차를 구한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
latest_year = df['Year'].max()
latest = df[df['Year'] == latest_year]
# idxmax 는 '최댓값이 있는 행의 라벨'을 준다 — .loc 에 넣으면 그 행 전체를 꺼낼 수 있다.
#  max() 는 값만 주므로 어느 나라인지 알 수 없다.
top_row = latest.loc[latest['Spending_USD'].idxmax()]
bottom_row = latest.loc[latest['Spending_USD'].idxmin()]
top_country = top_row['Country']
top_spending = top_row['Spending_USD']
bottom_country = bottom_row['Country']
bottom_spending = bottom_row['Spending_USD']
spending_gap = top_spending - bottom_spending
print(latest_year, top_country, round(top_spending, 2))
print(bottom_country, round(bottom_spending, 2), '격차', round(spending_gap, 2))

In [ ]:
# [자가채점]
assert latest_year == 2020
assert top_country == 'USA', '2020년 지출 1위는 미국이에요'
assert abs(top_spending - 11859.18) < 0.01
assert bottom_country == 'Japan', '2020년 지출 최하위는 일본이에요'
assert abs(bottom_spending - 4665.64) < 0.01
assert abs(spending_gap - 7193.54) < 0.01
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `df['Year'].max()` 로 최근 연도를 구해 그 연도만 필터한 뒤, `idxmax()`·`idxmin()` 이 돌려주는 **극값 행의 인덱스**를 `.loc[]` 에 넣어 1위·최하위 행 전체(나라·지출)를 꺼내고, 두 지출의 차로 격차를 구했습니다.
- **흔한 실수**: `max()`(값)와 `idxmax()`(인덱스)를 헷갈리기 쉽습니다. 나라 이름까지 알려면 값이 아니라 **행 인덱스**가 필요해요.
- **읽어 내기**: 2020년 1위 미국(11859.18)과 최하위 일본(4665.64)의 격차가 약 7193.54로, 미국이 최하위 나라의 2.5배 넘게 지출합니다 — 같은 해에도 나라별 의료비 지출 수준 차이가 매우 큽니다. 여섯 나라 지출이 모두 달라 동점 걱정은 없습니다.

## 4. 지출과 기대수명 산점도 + 관찰 (관계 + 색 구분 + 해석)
**배경**: 1인당 의료비 지출(`Spending_USD`)이 클수록 기대수명(`Life_Expectancy`)이 어떻게 움직이는지, 나라(`Country`)별로 색을 달리해 그리고, **그림에서 무엇이 보이는지**까지 읽어 봅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Spending_USD`(x)와 `Life_Expectancy`(y)의 **산점도**(`sns.scatterplot`)를 그리되 `hue='Country'` 로 나라별 색을 나누세요. 결과 Axes 를 `ax` 에 저장하고 제목을 `set_title` 으로 다세요.
- 그래프는 자가채점이 없습니다. 그린 뒤 아래 **서술 셀**에, 점들이 **어느 방향**으로 늘어서는지와 **오른쪽 위(고지출·고수명)** 에 주로 어느 나라가 있는지를 **1~2문장**으로 적으세요.

**예시**
```
x축: Spending_USD,  y축: Life_Expectancy
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 수치 변수의 관계는 산점도로 본다. 점이 어느 방향으로 늘어서는지 눈으로 읽는다.
- 세 번째 정보(나라)는 hue 로 색을 나눠 겹쳐 본다.

세부구현:
1. healthexp 를 다시 불러온다
2. scatterplot 으로 x 는 Spending_USD, y 는 Life_Expectancy 를 찍되 hue 에 Country 를 준다
3. 결과 Axes 를 ax 에 담고 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv2_q4.png" width="560"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
plt.figure(figsize=(8, 6))
# 나라별로 색을 나눈다 — 전체 점구름이 아니라 나라 안의 흐름을 봐야 관계가 제대로 보인다.
ax = sns.scatterplot(data=df, x='Spending_USD', y='Life_Expectancy', hue='Country')
ax.set_title('지출과 기대수명의 관계')
plt.show()

**관찰 (모범 서술 — 예시)**

점들은 전체적으로 **오른쪽 위로 늘어서** 지출이 클수록 기대수명도 함께 높아지는 양(+)의 흐름을 보인다. 특히 **미국**은 오른쪽 끝(가장 높은 지출)에 멀리 떨어져 있는데도 기대수명은 다른 나라보다 특별히 높지 않고, **일본**은 지출이 낮은 편인데도 기대수명이 가장 높은 축에 있어 — 지출이 늘 때 기대수명 증가폭이 점점 완만해짐을 짐작하게 한다.

### 해설 — 문제 4
- **접근법**: 산점도는 두 숫자 열을 x·y 좌표로 점 찍어 관계를 봅니다. 점들이 오른쪽 위로 늘어설수록 지출이 늘 때 기대수명도 함께 오르는 흐름이에요. `hue='Country'` 로 나라를 색으로 구분했습니다.
- **흔한 실수**: 축 이름은 seaborn 이 컬럼명으로 자동으로 답니다 — 굳이 `set_xlabel` 로 바꾸지 마세요.
- **관찰 채점 포인트**: 1) 점이 오른쪽 위로 늘어서는 양의 관계를 읽었는지 2) 미국(고지출·상대적 저효율)이나 일본(저지출·고수명) 같은 눈에 띄는 나라를 짚었는지를 봅니다. 정답은 하나가 아니며 그림에서 실제로 보이는 사실이면 됩니다.

## 5. 지출과 기대수명의 상관계수 (피어슨 + 해석)
**배경**: 산점도에서 눈으로 본 관계를 숫자 하나로 요약합니다. 피어슨 상관계수로 두 변수가 **직선으로** 얼마나 함께 움직이는지 잽니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `stats.pearsonr` 로 `Spending_USD` 와 `Life_Expectancy` 의 상관계수를 구해 `pearson_r` 에 담으세요(반환값의 **첫 번째**가 상관계수입니다).
- 상관계수 절댓값을 기준으로 강도를 판단해 `pearson_strength` 에 담으세요: 0.3 미만이면 `'약함'`, 0.3 이상 0.7 미만이면 `'중간'`, 0.7 이상이면 `'강함'`.
- `pearson_r` 은 소수 **셋째 자리**까지 비교합니다.
- 자가채점 아래 **서술 셀**에, 지출과 기대수명이 어느 방향으로 얼마나 강하게 이어지는지 **1문장 이상** 적으세요.

**예시**
```
round(pearson_r, 3) → 0.579
pearson_strength    → '중간'
```
<details><summary>힌트</summary>

```text
접근방법:
- pearsonr 는 두 값을 돌려주므로 첫 번째(상관계수)만 쓴다.
- 강도는 절댓값을 세 구간(약함·중간·강함)으로 나눠 판단한다.

세부구현:
1. pearsonr 에 두 열을 넣어 첫 번째 반환값을 pearson_r 에 담는다
2. pearson_r 의 절댓값을 0.3·0.7 을 경계로 조건 분기해 약함/중간/강함 문자열을 정한다
3. 그 문자열을 pearson_strength 에 담는다
```

</details>

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
# 두 번째 반환값은 p 값이지만 여기서는 세기만 보므로 _ 로 버린다.
pearson_r, _ = stats.pearsonr(df['Spending_USD'], df['Life_Expectancy'])
if abs(pearson_r) < 0.3:
    pearson_strength = '약함'
elif abs(pearson_r) < 0.7:
    pearson_strength = '중간'
else:
    pearson_strength = '강함'
print(round(pearson_r, 3), pearson_strength)

In [ ]:
# [자가채점]
assert abs(pearson_r - 0.579) < 0.01
assert pearson_strength == '중간', '0.3 이상 0.7 미만이면 중간이에요'
print("✅ 문제5 통과!")

**관찰 (모범 서술 — 예시)**

지출(`Spending_USD`)이 늘수록 기대수명(`Life_Expectancy`)도 대체로 함께 오르는 양(+)의 관계이며, 상관계수 0.579로 그 세기는 중간 정도다.

### 해설 — 문제 5
- **접근법**: `stats.pearsonr(x, y)` 는 **두 값** 튜플을 돌려줍니다. 첫 번째(상관계수)만 받아 `pearson_r` 에 담고, 절댓값을 세 구간으로 나눠 강도를 분류했어요.
- **흔한 실수**: 반환값을 통째로 변수에 담으면 튜플이라 `round(...)` 에서 에러가 납니다. `r, _ = ...` 로 첫 번째만 꺼내세요.
- **대안**: 강도 경계(0.3·0.7)는 관례일 뿐 절대 기준은 아닙니다 — 분야마다 다르게 잡기도 합니다.

## 6. 순위로 본 상관 (스피어만 + 비교)
**배경**: 피어슨은 직선 관계를 재지만, 관계가 살짝 휘어 있으면 순위 기반의 스피어만 상관이 더 잘 잡아냅니다. 두 값을 비교합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `stats.spearmanr` 로 `Spending_USD` 와 `Life_Expectancy` 의 스피어만 상관계수를 구해 `spearman_rs` 에 담으세요(반환값의 **첫 번째**).
- `spearman_rs` 는 소수 **셋째 자리**까지 비교합니다.
- 자가채점 아래 **서술 셀**에, 스피어만 상관계수를 앞 문제의 피어슨 상관계수와 비교해 어느 쪽이 더 큰지와 그 의미를 **1문장 이상** 적으세요.

**예시**
```
round(spearman_rs, 3) → 0.747
```
<details><summary>힌트</summary>

```text
접근방법:
- spearmanr 도 두 값을 돌려주므로 첫 번째(상관계수)만 쓴다.
- 앞 문제의 피어슨 값과 크기를 비교한다.

세부구현:
1. spearmanr 에 두 열을 넣어 첫 번째 반환값을 spearman_rs 에 담는다
2. 소수 셋째 자리로 반올림해 확인한다
3. 피어슨(0.579)과 크기를 견줘 서술한다
```

</details>

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
# 지출이 늘수록 기대수명이 오르지만 점점 완만해진다 — 이런 곡선 관계는 순위로 재는 스피어만이 더 크게 나온다.
spearman_rs, _ = stats.spearmanr(df['Spending_USD'], df['Life_Expectancy'])
print(round(spearman_rs, 3))

In [ ]:
# [자가채점]
assert abs(spearman_rs - 0.747) < 0.01
print("✅ 문제6 통과!")

**관찰 (모범 서술 — 예시)**

스피어만 상관계수(0.747)가 피어슨 상관계수(0.579)보다 크다. 이는 지출과 기대수명이 정확한 직선은 아니어도 순위로 보면 꾸준히 함께 오르는 단조 관계가 더 강하다는 뜻으로, 지출이 커질수록 기대수명 증가폭이 완만해지는 휘어진 관계를 짐작하게 한다.

### 해설 — 문제 6
- **접근법**: `stats.spearmanr` 는 값 대신 **순위**로 상관을 잽니다. 순위 상관이 피어슨보다 크면 관계가 직선보다 곡선(단조 증가)에 가깝다는 신호예요.
- **흔한 실수**: 피어슨과 스피어만을 같은 값으로 기대하면 안 됩니다 — 관계가 휘어 있을수록 둘의 차이가 커집니다.
- **대안**: 두 상관을 나란히 보려면 `df[['Spending_USD', 'Life_Expectancy']].corr(method='spearman')` 처럼 `method` 인자를 바꿔 구할 수도 있습니다.

## 7. 표본평균의 분포 — 중심극한정리 (시뮬레이션 + 관찰)
**배경**: 원래 지출(`Spending_USD`) 분포는 한쪽으로 치우쳐 있습니다. 그런데 이 모집단에서 30개씩 뽑아 **평균**을 여러 번 구하면, 그 평균들의 분포는 어떤 모양이 될까요? 중심극한정리를 눈으로 확인합니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Spending_USD` 전체를 모집단으로 삼아, 크기 **30**인 표본을 **복원추출**해 그 평균을 구하는 일을 **2000번** 반복하고, 그 표본평균들을 `sample_means` 에 모으세요.
- 재현을 위해 맨 앞에서 `rng = np.random.default_rng(42)` 로 난수 생성기를 만들어 `rng.choice(모집단, size=30, replace=True)` 로 표본을 뽑으세요.
- `sample_means` 의 **히스토그램**을 `sns.histplot(sample_means, bins=30, kde=True)` 로 그려 결과 Axes 를 `ax` 에 저장하고 제목을 다세요.
- 그래프 자체는 채점하지 않지만, `sample_means` 는 자가채점합니다. 아래 **서술 셀**에도 원래 지출 분포의 모양과 표본평균 분포의 모양이 어떻게 다른지 **2문장 이상** 관찰을 적으세요.

**예시**
```
len(sample_means) → 2000
x축: 표본평균,  종 모양(정규분포)에 가까운 분포
```
<details><summary>힌트</summary>

```text
접근방법:
- 모집단에서 표본을 뽑아 평균을 구하는 일을 아주 여러 번 반복하고, 그 평균들을 모아 히스토그램으로 그린다.
- 복원추출은 rng.choice 에 replace=True 로 준다.

세부구현:
1. Spending_USD 값 전체를 모집단 배열로 꺼낸다
2. 반복문(또는 리스트 컴프리헨션)으로 2000번, 매번 크기 30 표본을 복원추출해 평균을 구해 모은다
3. 모은 표본평균들을 histplot 으로 그리고 ax 에 담아 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv2_q7.png" width="560"/>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
rng = np.random.default_rng(42)
pop = df['Spending_USD'].values
# 30개를 뽑아 평균 내기를 2000번 되풀이한다. replace=True 라야 매번 같은 모집단에서 새로 뽑는 셈이 된다.
#  원본이 치우쳐 있어도 '표본평균들'의 분포는 종 모양에 가까워진다 — 중심극한정리다.
sample_means = [rng.choice(pop, size=30, replace=True).mean() for _ in range(2000)]
plt.figure(figsize=(8, 5))
ax = sns.histplot(sample_means, bins=30, kde=True)
ax.set_title('표본평균 2000개의 분포 (n=30)')
ax.set_xlabel('표본평균')
plt.show()

In [ ]:
# [자가채점]  그래프는 채점하지 않고, 모은 표본평균 2000개만 확인합니다.
assert len(sample_means) == 2000
assert abs(np.mean(sample_means) - 2789.34) < 60, '표본평균들의 중심은 모평균 부근이어야 합니다'
assert 250 < np.std(sample_means) < 550, '표본평균들의 흩어짐이 이론값(σ/√30 ≈ 401)과 크게 어긋납니다'
print("✅ 문제7 통과!")

**관찰 (모범 서술 — 예시)**

원래 지출 분포는 값이 작은 쪽에 몰리고 오른쪽으로 길게 늘어진 비대칭 모양이지만, 30개씩 뽑은 표본평균 2000개의 분포는 좌우 대칭의 종 모양(정규분포)에 가깝다. 또 표본평균들은 모평균 부근에 좁게 모여 있어, 개별 값보다 평균이 훨씬 덜 흔들린다는 중심극한정리를 보여 준다.

### 해설 — 문제 7
- **접근법**: 모집단에서 크기 30 표본을 복원추출해 평균을 구하는 일을 2000번 반복하면 **표본평균의 분포(표집분포)** 가 만들어집니다. 원분포가 치우쳐 있어도 표본평균 분포는 종 모양으로 모입니다.
- **흔한 실수**: `replace=False`(비복원)로 두면 표본 크기가 모집단에 가까워질 때 다양성이 줄어듭니다. 여기서는 `replace=True` 로 복원추출하세요. 난수 생성기를 안 고정하면 그림이 매번 조금씩 달라집니다(모양은 같습니다).
- **대안**: `size=30` 을 `100` 으로 키우면 표본평균 분포가 더 좁아져(표준오차 감소) 중심에 더 뾰족하게 모입니다.

## 8. 기대수명 평균의 95% 신뢰구간 (공식)
**배경**: 표본에서 구한 평균 하나만으로는 참 평균을 알 수 없습니다. 평균이 어느 범위 안에 있을지 **95% 신뢰구간**으로 나타냅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Life_Expectancy` 전체 평균에 대한 **95% 신뢰구간**을 구해 하한을 `ci_low`, 상한을 `ci_high` 에 담으세요.
- 신뢰구간은 **표본평균을 중심으로 표준오차만큼 좌우로 벌린 범위**입니다. `scipy.stats` 에서 표준오차를 구하는 함수(`sem`)와, 신뢰수준·평균(`loc`)·표준오차(`scale`)를 받아 (하한, 상한)을 돌려주는 정규분포 구간 함수(`norm.interval`)를 쓰세요.
- **주의**: `scale` 에 표준편차가 아니라 **표준오차**를 넣어야 *평균*의 신뢰구간이 됩니다.
- 두 값 모두 소수 **둘째 자리**까지 비교합니다.
- **주의**: 이 데이터는 국가·연도가 섞여 있어 완전히 독립 표본은 아니지만, 이 문제에서는 독립으로 가정하고 신뢰구간 공식을 연습합니다(실제로는 구간이 더 넓어집니다).

**예시**
```
round(ci_low, 2)  → 77.52
round(ci_high, 2) → 78.3
```
<details><summary>힌트</summary>

```text
접근방법:
- 신뢰구간은 평균을 중심으로 표준오차만큼 좌우로 벌린 범위다.
- norm.interval 에 신뢰수준·평균(loc)·표준오차(scale)를 넣으면 (하한, 상한)을 돌려준다.

세부구현:
1. Life_Expectancy 의 평균과 표준오차(stats.sem)를 구한다
2. 정규분포 구간 함수에 신뢰수준 0.95 와 평균·표준오차를 넣어 호출한다
3. 돌려받은 두 값을 각각 ci_low, ci_high 에 담는다
```

</details>

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
life = df['Life_Expectancy']
# scale 에는 표준편차가 아니라 표준오차(sem)를 넣는다 — 구간이 재는 것은 '평균'의 흔들림이다.
ci_low, ci_high = stats.norm.interval(0.95, loc=life.mean(), scale=stats.sem(life))
print(round(ci_low, 2), round(ci_high, 2))

In [ ]:
# [자가채점]
assert abs(ci_low - 77.52) < 0.01
assert abs(ci_high - 78.3) < 0.01
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: 신뢰구간은 `평균 ± (임계값 × 표준오차)` 입니다. `stats.sem` 이 표준오차(표준편차 ÷ √n)를, `stats.norm.interval` 이 정규분포 임계값을 써서 하한·상한을 한 번에 돌려줍니다.
- **흔한 실수**: `scale` 에 표준편차(`.std()`)를 넣으면 안 됩니다 — **표준오차**(`stats.sem`)를 넣어야 평균의 신뢰구간이 됩니다.
- **독립 가정 주의**: `stats.sem` 은 274개 행이 서로 독립인 것처럼 표준오차를 √274 로 나눕니다. 실제로는 같은 나라 안에서 인접 연도끼리 값이 닮아 있어(자기상관) 완전한 독립 표본이 아니므로, 실제 표준오차는 이보다 크고 신뢰구간도 더 넓어질 수 있습니다. 여기서는 신뢰구간 공식을 연습하기 위해 독립으로 가정했습니다.
- **해석 주의**: '95% 신뢰구간 [77.52, 78.3]'은 참평균이 그 구간에 있을 확률이 95%라는 뜻이 아니라, **같은 방식으로 표본을 반복해 구간을 아주 많이 만들면 그중 약 95%가 진짜 모평균을 포함**한다는 뜻입니다(교안 참고).
- **대안**: 표본이 작을 때는 정규분포 대신 t분포(`stats.t.interval(0.95, df=n-1, ...)`)를 씁니다. 여기서는 표본이 커 두 결과가 거의 같습니다.

## 9. 두 주장을 p-value 로 견주기 (+ 신뢰구간과의 관계)
**배경**: 신뢰구간은 어떤 주장이 **안이냐 밖이냐**만 답합니다. 그런데 밖에 있는 주장들끼리도 **얼마나** 어긋났는지는 서로 다르죠. 그 어긋난 **정도**를 숫자 하나로 재는 것이 **p-value** 입니다. 두 보고서의 주장을 p-value 로 견주고, 그 결과가 문제 8 의 신뢰구간과 **일치하는지** 확인합니다.

- 보고서 A: "이 나라들의 평균 기대수명은 **78.5세** 다"
- 보고서 B: "아니다, **78.0세** 다"

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Life_Expectancy` 의 표본평균과 표준오차(`stats.sem`)로, 각 주장에 대한 **t 통계량**을 `t = (표본평균 − 주장값) / 표준오차` 로 구하세요.
- **양측 p-value** 를 `2 * stats.t.sf(abs(t), df=n-1)` 로 구해 각각 `p_a`(주장 78.5)·`p_b`(주장 78.0) 에 담으세요(`n` 은 값의 개수, 양측이라 **2를 곱합니다**).
- 두 p 값 모두 소수 **넷째 자리**까지 비교합니다.
- 각 주장이 문제 8 의 95% 신뢰구간 `[77.52, 78.30]` 안에 있는지 판단해, 문자열 `'구간 밖'` 또는 `'구간 안'` 을 `verdict_a`·`verdict_b` 에 담으세요.
- 자가채점 아래 **서술 셀**에, **p < 0.05 인 주장과 신뢰구간 밖인 주장이 서로 일치하는지**를 **1~2문장**으로 적으세요.

**예시**
```
round(p_a, 4) → 0.0031     verdict_a → '구간 밖'
round(p_b, 4) → 0.6478     verdict_b → '구간 안'
```
<details><summary>힌트</summary>

```text
접근방법:
- 주장값에서 표본평균이 표준오차 몇 칸만큼 떨어졌는지가 t 통계량이다.
- 그 t 가 우연히 나올 확률을 t분포의 꼬리 넓이로 재고, 양쪽 꼬리를 보므로 2배 한다.

세부구현:
1. Life_Expectancy 의 평균·표준오차(stats.sem)·개수를 구한다
2. 주장값마다 t 를 계산하고 stats.t.sf 에 abs(t) 와 자유도를 넣어 한쪽 꼬리 넓이를 구한 뒤 2배 한다
3. 주장값이 [77.52, 78.30] 사이에 있는지 비교해 판정 문자열을 담는다
```

</details>

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
life = df['Life_Expectancy']
xbar, se, n = life.mean(), stats.sem(life), len(life)

# 신뢰구간과 검정은 같은 것을 다르게 말한 것이다 — 구간 밖의 값은 p < 0.05, 안의 값은 p > 0.05 가 된다.
t_a = (xbar - 78.5) / se
t_b = (xbar - 78.0) / se
p_a = 2 * stats.t.sf(abs(t_a), df=n - 1)
p_b = 2 * stats.t.sf(abs(t_b), df=n - 1)

verdict_a = '구간 안' if 77.52 <= 78.5 <= 78.30 else '구간 밖'
verdict_b = '구간 안' if 77.52 <= 78.0 <= 78.30 else '구간 밖'
print(round(p_a, 4), verdict_a)
print(round(p_b, 4), verdict_b)

In [ ]:
# [자가채점]
assert abs(p_a - 0.0031) < 0.0005
assert abs(p_b - 0.6478) < 0.0005
assert verdict_a == '구간 밖'
assert verdict_b == '구간 안'
assert (p_a < 0.05) == (verdict_a == '구간 밖')   # 구간 밖 ⟺ p < 0.05
assert (p_b < 0.05) == (verdict_b == '구간 밖')
print("✅ 문제9 통과!")

**관찰 (모범 서술 — 예시)**

주장 78.5 는 p = 0.0031 로 0.05 보다 훨씬 작고, 동시에 95% 신뢰구간 [77.52, 78.30] 의 **밖**에 있다. 반대로 주장 78.0 은 p = 0.6478 로 크고 구간 **안**에 있다. 즉 '신뢰구간 밖'과 'p < 0.05' 는 정확히 같은 판단을 다르게 표현한 것이며, p-value 는 거기에 더해 **얼마나 어긋났는지의 정도**까지 알려 준다.

### 해설 — 문제 9
- **접근법**: p-value 는 "주장이 옳다고 가정했을 때, 지금 본 표본평균만큼(또는 그보다 더) 극단적인 값이 우연히 나올 확률"입니다. 표본평균이 주장값에서 표준오차 몇 칸 떨어졌는지(t)를 구하고, 그 t 바깥쪽 꼬리 넓이를 `stats.t.sf` 로 잽니다.
- **왜 2를 곱하나**: "78.5 와 다르다"는 주장은 **더 클 수도, 더 작을 수도** 있으니 양쪽 꼬리를 모두 셉니다(양측). 한쪽만 세면 p 가 절반으로 나와 실제보다 주장을 쉽게 의심하게 됩니다.
- **구간 ⟺ p 의 등가**: 95% 신뢰구간의 **경계에서 정확히 p = 0.05** 가 됩니다. 그래서 구간 밖이면 p < 0.05, 구간 안이면 p > 0.05 로 항상 맞물립니다. 자가채점의 마지막 두 줄이 이 등가를 직접 확인합니다.
- **오해 주의**: p = 0.0031 은 "주장이 틀릴 확률이 99.7%"라는 뜻이 **아닙니다**. 어디까지나 "주장이 맞다고 가정했을 때 이런 결과가 나올 확률"입니다.
- **다음 단원 예고**: 여기서는 t 를 손으로 계산했지만, 다음 시간에는 `ttest` 계열 함수가 이 과정을 한 줄로 처리해 줍니다. 지금 손으로 만들어 본 덕분에 그 함수가 무엇을 돌려주는지 알고 쓰게 됩니다.

## 10. 세 변수 상관행렬 히트맵 + 관찰 (상관 + 색칠 + 해석)
**배경**: 연도·지출·기대수명 세 수치 변수가 서로 얼마나 함께 움직이는지 **상관행렬**로 구하고, 히트맵으로 한눈에 본 뒤, **어느 두 변수의 관계가 가장 강한지**까지 읽어 봅니다.

**요구사항**:
- `data/healthexp.csv` 를 읽어 변수 `df` 에 담으세요.
- `Year`·`Spending_USD`·`Life_Expectancy` 세 열의 **상관행렬**을 `df[[...]].corr()` 로 구해 `corr` 에 담으세요(shape `(3, 3)`).
- `corr` 를 `sns.heatmap(corr, annot=True, cmap='Blues')` 로 그려 숫자를 칸에 표시하세요. 결과 Axes 를 `ax` 에 저장하고 제목을 다세요.
- 그래프는 자가채점이 없습니다. 그린 뒤 아래 **서술 셀**에, 대각선(자기 자신, 값 1)을 빼고 **가장 큰 상관을 보인 두 변수의 짝**과 그 값을 **1~2문장**으로 적으세요.

**예시**
```
corr.shape → (3, 3)
x·y축: Year, Spending_USD, Life_Expectancy
```
<details><summary>힌트</summary>

```text
접근방법:
- 여러 수치 열의 상관을 한 번에 보려면 corr 로 상관행렬을 만들고, 그 표를 heatmap 으로 색칠한다.

세부구현:
1. 세 열만 뽑아 corr 로 상관행렬 corr 를 만든다
2. heatmap 으로 corr 를 그리되 annot=True 로 칸마다 숫자를 적는다
3. 결과 Axes 를 ax 에 담고 제목을 단다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv2_q10.png" width="560"/>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../day08_기술통계_추론통계/data/healthexp.csv')
corr = df[['Year', 'Spending_USD', 'Life_Expectancy']].corr()
plt.figure(figsize=(6, 5))
# 세 변수만 골라 본다 — 관계없는 열까지 넣으면 표만 커지고 읽기 어려워진다.
ax = sns.heatmap(corr, annot=True, cmap='Blues')
ax.set_title('연도·지출·기대수명 상관행렬')
plt.show()

**관찰 (모범 서술 — 예시)**

대각선(값 1)을 빼면 **연도(`Year`)와 기대수명(`Life_Expectancy`)** 의 상관이 약 **0.90** 으로 가장 강하다. 연도와 지출(0.83)도 강하고, 정작 지출과 기대수명(0.58)은 그보다 약해 — 시간이 흐르며 지출과 기대수명이 각각 함께 커졌을 뿐, 지출이 곧바로 수명을 끌어올린 것은 아닐 수 있음을 시사한다.

### 해설 — 문제 10
- **접근법**: `df[[...]].corr()` 가 열끼리의 피어슨 상관을 모두 계산해 정사각 행렬을 만들고, `heatmap(annot=True)` 가 그 값을 색과 숫자로 함께 보여 줍니다. 대각선은 자기 자신과의 상관이라 항상 1이에요.
- **흔한 실수**: `corr()` 는 수치형 열끼리만 계산합니다. 문자열 열(`Country`)이 섞여 있으면 자동으로 빠지지만, 필요한 세 열만 명시적으로 골라 넣는 편이 깔끔합니다.
- **관찰 채점 포인트**: 대각선(1)을 제외하고 가장 큰 값이 연도↔기대수명(약 0.90)임을 짚었는지를 봅니다. 연도가 지출·기대수명과 강하게 상관되는 것은 시간이 흐르며 둘 다 함께 커졌기 때문입니다.

## 11. 인사이트 리포트 (종합 서술)
**배경**: 지금까지 구한 상관계수와 신뢰구간을 근거로, "의료비 지출과 기대수명은 어떤 관계인가"를 한 문단으로 정리합니다.

**요구사항**:
- 앞 문제들에서 구한 **상관계수(피어슨·스피어만)** 와 **기대수명 평균의 95% 신뢰구간**을 근거로 들며, 지출과 기대수명의 관계를 **3문장 이상**으로 아래 서술 셀에 적으세요.
- 상관은 인과가 아니라는 점(지출이 많다고 반드시 더 오래 사는 것은 아님)도 한 문장 포함하면 좋습니다.

**인사이트 리포트 (모범 서술 — 예시)**

여섯 나라의 데이터에서 1인당 의료비 지출과 기대수명은 함께 오르는 양(+)의 관계를 보인다. 피어슨 상관계수는 0.579로 직선 관계는 중간 정도지만, 순위로 본 스피어만 상관계수는 0.747로 더 커서 지출이 늘수록 기대수명이 꾸준히 오르되 그 증가폭은 점차 완만해지는 곡선 형태임을 시사한다. 기대수명 전체 평균의 95% 신뢰구간은 약 77.52~78.30세다 — 이는 이 구간에 참평균이 있을 확률이 95%라는 뜻이 아니라, 같은 방식으로 표본을 반복해 구간을 아주 많이 만들면 그중 95%가 모평균을 포함한다는 뜻이다(다만 이 데이터는 국가·연도가 섞여 완전히 독립은 아니라 실제 구간은 이보다 넓을 수 있다). 다만 상관은 인과가 아니므로, 지출을 늘린다고 해서 곧바로 기대수명이 늘어난다고 단정할 수는 없으며 소득·의료 접근성 같은 다른 요인도 함께 작용한다.

### 해설 — 문제 11
- **평가 포인트**: 1) 방향(양의 관계) 2) 세기(피어슨 0.579·스피어만 0.747, 순위 상관이 더 큼) 3) 신뢰구간(약 77.52~78.30세)을 근거로 들었는지 4) 상관≠인과를 언급했는지를 봅니다.
- **흔한 실수**: 숫자 없이 "관계가 있다"만 적으면 근거가 약합니다. 앞에서 구한 값을 인용하도록 지도하세요.